In [2]:
# 1. 파이썬 코드에서 Matplotlib 폰트 설정
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import numpy as np
from datetime import timedelta
from itertools import combinations
from collections import Counter

# 폰트 설정
plt.rc('font', family='Malgun Gothic')
# 마이너스 부호 깨짐 방지
plt.rcParams['axes.unicode_minus'] = False

print("한글 폰트 설정이 완료되었습니다.")

# 3. 데이터 로드 및 통합 (모든 문제 풀이의 시작점)
try:
    orders_df = pd.read_csv('../data/orders.csv')
    payments_df = pd.read_csv('../data/payments.csv')
    products_df = pd.read_csv('../data/products.csv')
    shipping_df = pd.read_csv('../data/shipping.csv')
    customers_df = pd.read_csv('../data/customers.csv')

    # 모든 데이터프레임 병합
    df = pd.merge(orders_df, payments_df, on='order_id', how='left')
    df = pd.merge(df, products_df, on='product_id', how='left')
    df = pd.merge(df, customers_df, on='customer_id', how='left')
    df = pd.merge(df, shipping_df, on='order_id', how='left')

    # 데이터 전처리
    date_cols = ['order_date', 'payment_date', 'join_date', 'shipping_start_date', 'shipping_end_date']
    for col in date_cols:
        df[col] = pd.to_datetime(df[col], errors='coerce')
    df['total_sales'] = df['quantity'] * df['price']
    
    print("데이터 로드 및 통합이 완료되었습니다.")

except FileNotFoundError as e:
    print(f"파일을 찾을 수 없습니다: {e}")


한글 폰트 설정이 완료되었습니다.
데이터 로드 및 통합이 완료되었습니다.


In [3]:
# 문제 7: 고객의 가입 후 첫 구매까지 걸리는 평균 시간을 계산하세요.
# 비즈니스 목적: 고객 활성화까지의 기간(Lead Time)을 측정하여, 신규 가입 고객을 위한 온보딩 프로세스(가입 환영 쿠폰, 사용 가이드 등)의 효과를 평가합니다.

# 출력 결과를 보고 코딩하세요


가입 후 첫 구매까지의 평균 시간: -42 days +06:19:37.922110553

In [7]:
# =====================================================
# [현재 문제에서 필요한 컬럼 설명]
# join_date : 고객 가입일 (datetime 형식)
# order_date : 고객 주문일 (datetime 형식)
# customer_id : 고객 구분을 위한 ID
# 
# 🚨 현재 df에는 '첫 구매일(first purchase)' 컬럼이 없음
# → 따라서 고객별 첫 주문일을 직접 계산해서 만들어야 함
# =====================================================

# (1) join_date와 order_date가 datetime 형식인지 다시 확인
df['join_date'] = pd.to_datetime(df['join_date'], errors='coerce')    # 고객 가입일을 날짜형으로 변환
df['order_date'] = pd.to_datetime(df['order_date'], errors='coerce')  # 고객 주문일을 날짜형으로 변환

# (2) 고객별 '첫 구매일' 계산
# groupby('customer_id') : 고객별로 데이터 묶기
# ['order_date'].min() : 각 고객의 주문 날짜 중 가장 빠른 날짜 선택 = 첫 구매일
first_purchase_df = df.groupby('customer_id')['order_date'].min().reset_index()

# (3) 컬럼 이름을 명확하게 변경
first_purchase_df.rename(columns={'order_date': 'first_purchase_date'}, inplace=True)

# (4) 원본 df의 고객 가입일(join_date)과 첫 구매일(first_purchase_date)을 merge하여 하나로 합치기
merged_df = pd.merge(df[['customer_id', 'join_date']].drop_duplicates(),
                     first_purchase_df,
                     on='customer_id',
                     how='left')

# (5) 가입 → 첫 구매까지 걸린 시간(Timedelta) 계산
# first_purchase_date - join_date : 날짜끼리 빼면 '시간 차이(Timedelta)' 자동 계산됨
merged_df['lead_time'] = merged_df['first_purchase_date'] - merged_df['join_date']

# (6) 평균 리드타임 계산
average_lead_time = merged_df['lead_time'].mean()


print("가입 후 첫 구매까지의 평균 시간:", average_lead_time)



가입 후 첫 구매까지의 평균 시간: -42 days +06:19:37.922110553


In [ ]:
# 🧩 2️⃣ columns: 내용 분석

# 지금 이 리스트는 df 안의 모든 컬럼(속성) 이름이에요.

# 컬럼 이름	역할 설명
# order_id	주문 고유번호
# customer_id	고객 고유번호
# product_id	제품 고유번호
# order_date	주문한 날짜
# quantity	주문 수량
# payment_method	결제 방식 (카드, 계좌이체 등)
# payment_status	결제 상태 (완료, 취소 등)
# payment_date	결제일
# product_name	제품 이름
# category	상품 카테고리
# price	상품 가격 (단가)
# stock	재고 수량
# name	고객 이름
# gender	고객 성별
# age	고객 나이
# join_date	고객 가입일
# city	고객 도시
# shipping_id	배송 고유번호
# shipping_company	배송사 이름
# shipping_status	배송 상태
# shipping_start_date	배송 시작일
# shipping_end_date	배송 완료일
# total_sales	총 주문 금액 (quantity × price) — ✅ 우리가 분석에 쓸 핵심 컬럼
# age_group	연령대 (예: 10대, 20대 …) — ✅ 히트맵 만들 때 꼭 필요